Una subclase es una clase nueva que parte de otra (la clase base o padre):
- hereda todos sus atributos y métodos,
- puede agregar métodos nuevos,
- puede redefinir métodos que ya existían.

Se escribe poniendo la clase base entre paréntesis: `class Hija(Padre)`

In [1]:
class Observacion:
    def __init__(self, ciudad, temperatura):
        self.ciudad = ciudad
        self.temperatura = temperatura

    def temperatura_f(self):
        return self.temperatura * 9 / 5 + 32

    def __str__(self):
        return f"{self.ciudad}: {self.temperatura} °C"

    def __repr__(self):
        return f"Observacion({self.ciudad!r}, {self.temperatura!r})"
        

In [3]:
obj = Observacion("comayo", "30")
obj

Observacion('comayo', '30')

In [4]:
class ObservacionDeMontana(Observacion):
    def es_bajo_cero(self):          # método nuevo
        return self.temperatura < 0


o = ObservacionDeMontana("Cerro Catedral", -3.5)
print(o)                     # __str__ heredado
print(o.temperatura_f())     # método heredado
print(o.es_bajo_cero())      # método nuevo

Cerro Catedral: -3.5 °C
25.7
True


`ObservacionDeMontana` no tiene `__init__` propio: usa el de `Observacion`.

### ¿Dónde busca Python un método?

Primero en la clase del objeto; si no lo encuentra, en la clase base; y así hacia arriba.

In [2]:
print("es_bajo_cero" in ObservacionDeMontana.__dict__)   # definido acá
print("temperatura_f" in ObservacionDeMontana.__dict__)  # no: lo hereda
print(ObservacionDeMontana.__bases__)

True
False
(<class '__main__.Observacion'>,)


## Redefinir métodos (override)

Si la subclase define un método con el mismo nombre, reemplaza al heredado.

In [ ]:
class ObservacionDeMontana(Observacion):
    def es_bajo_cero(self):
        return self.temperatura < 0

    def __str__(self):                               # redefinido
        aviso = " ❄ helada" if self.es_bajo_cero() else ""
        return f"{self.ciudad}: {self.temperatura} °C{aviso}"

    def __repr__(self):
        return f"ObservacionDeMontana({self.ciudad!r}, {self.temperatura!r})"


obs = [Observacion("Neuquén", 21.0), ObservacionDeMontana("Cerro Catedral", -3.5)]
for o in obs:
    print(o)        # cada uno usa SU __str__
obs

### agregar atributos: redefinir `__init__`

Si la subclase necesita un atributo más, puede tener su propio `__init__`.
Reemplaza por completo al del padre, así que hay que volver a agregar los atributos.

In [ ]:
class ObservacionConViento(Observacion):
    def __init__(self, ciudad, temperatura, viento_kmh):
        self.ciudad = ciudad              # repetido del padre
        self.temperatura = temperatura    # repetido del padre
        self.viento_kmh = viento_kmh      # nuevo

    def __str__(self):
        return f"{self.ciudad}: {self.temperatura} °C, viento {self.viento_kmh} km/h"


ov = ObservacionConViento("Ushuaia", 3.2, 55)
print(ov)
print(ov.temperatura_f())    # lo heredado sigue funcionando

### `super()`: reusar el método del padre

En vez de repetir lo que ya hace el `__init__` del padre, lo podemos llamar con `super()`.
`super()` devuelve "la clase base" de este objeto, así que `super().__init__(...)` ejecuta el
`__init__` de `Observacion` y después agregamos solo lo nuevo.

Ojo: a `super().__init__(...)` no se le pasa `self`. `super()` ya está "atado" al objeto, igual que
cuando llamamos `ov.temperatura_f()` sin pasar `ov`. Si escribimos `super().__init__(self, ciudad, temperatura)`
Python recibe un argumento de más y da `TypeError`.

In [ ]:
class ObservacionConViento(Observacion):
    def __init__(self, ciudad, temperatura, viento_kmh):
        super().__init__(ciudad, temperatura)   # el padre asigna ciudad y temperatura
        self.viento_kmh = viento_kmh            # nuevo


ov = ObservacionConViento("Ushuaia", 3.2, 55)
print(ov.ciudad, ov.temperatura, ov.viento_kmh)
print(ov)    # __str__ heredado: no muestra el viento


`super()` sirve con cualquier método, no solo con `__init__`. Es útil para extender un método
del padre en lugar de reemplazarlo por completo: usamos lo que devuelve y le agregamos algo.

In [ ]:
class ObservacionConViento(Observacion):
    def __init__(self, ciudad, temperatura, viento_kmh):
        super().__init__(ciudad, temperatura)
        self.viento_kmh = viento_kmh

    def __str__(self):
        return super().__str__() + f", viento {self.viento_kmh} km/h"


ov = ObservacionConViento("Ushuaia", 3.2, 55)
print(ov)
print(Observacion("Neuquén", 21.0))   # el padre no cambió


## Agregar funcionalidades a clases built-in

`list`, `dict`, `str`, `float`… son clases, así que podemos crear clases hijas de ellas.


In [ ]:
class Serie(list):
    def promedio(self):
        return sum(self) / len(self)

    def amplitud(self):
        return max(self) - min(self)

    def validos(self):
        """Devuelve una Serie nueva sin los None."""
        return Serie(filter(lambda x: x is not None, self))

    def a_fahrenheit(self):
        return Serie(map(lambda c: c * 9 / 5 + 32, self))


temps = Serie([8.5, 10.1, None, 12.3, 7.9])
temps.append(9.4)              # método de list
print(len(temps), temps[0])    # todo lo de list sigue andando

v = temps.validos()
print(v, type(v).__name__)
print(round(v.promedio(), 2), round(v.amplitud(), 2))
print(v.a_fahrenheit())

Sigue siendo una lista en todo sentido:

In [ ]:
print(isinstance(v, list))
import json
print(sorted(v))
print(json.dumps(v))

Trampa: los métodos heredados que crean una lista nueva devuelven una `list` común, no una `Serie`.

In [ ]:
print(type(v[1:3]).__name__)    # el slicing devuelve list
print(type(v + v).__name__)     # la suma también

# v[1:3].promedio()   # AttributeError: 'list' object has no attribute 'promedio'
print(Serie(v[1:3]).promedio()) # solución: envolver de nuevo

Por eso en `validos` y `a_fahrenheit` devolvimos `Serie(...)` explícitamente.

### Un diccionario de ciudades

Exactamente la estructura del proyecto: clave = ciudad, valor = diccionario con los datos.

In [ ]:
import json

class Ciudades(dict):
    def top(self, campo, n=3, mayor=True):
        """Las n ciudades con mayor (o menor) valor de campo."""
        ordenadas = sorted(self, key=lambda c: self[c][campo], reverse=mayor)
        return ordenadas[:n]

    def con_campo(self, campo):
        """Sub-diccionario con las ciudades que tienen ese campo."""
        return Ciudades({c: d for c, d in self.items() if d.get(campo) is not None})

    def guardar_json(self, ruta):
        with open(ruta, "w", encoding="utf-8") as f:
            json.dump(self, f, indent=2, ensure_ascii=False)


datos = Ciudades({
    "Bariloche": {"temperatura": 8.5, "viento": 25, "st": 5.0},
    "Neuquén":   {"temperatura": 21.0, "viento": 40, "st": None},
    "Ushuaia":   {"temperatura": 3.2, "viento": 55, "st": -2.1},
    "Mendoza":   {"temperatura": 24.1, "viento": 10, "st": None},
})

print(datos.top("temperatura", 2))
print(datos.top("viento", 2, mayor=False))
print(list(datos.con_campo("st")))
print(datos["Bariloche"], len(datos), "Mendoza" in datos)   # sigue siendo un dict

In [ ]:
datos.guardar_json("ciudades.json")
with open("ciudades.json", encoding="utf-8") as f:
    print(f.read()[:120], "...")

`json.dump` lo acepta sin problema porque es un `dict`. Al leerlo de nuevo vuelve como `dict` común,
y lo convertimos:

In [ ]:
with open("ciudades.json", encoding="utf-8") as f:
    leido = Ciudades(json.load(f))
leido.top("temperatura", 1)

### Un diccionario que cuenta: el método especial `__missing__`

`dict` llama a `__missing__` cuando se pide una clave que no existe (en vez de tirar `KeyError`),
pero solo si lo definimos en una subclase.

In [ ]:
class Contador(dict):
    def __missing__(self, clave):
        return 0

direcciones = ["Oeste", "Norte", "Oeste", "Sur", "Oeste", "Norte"]
cuenta = Contador()
for d in direcciones:
    cuenta[d] += 1        # la primera vez cuenta[d] vale 0

print(cuenta)
print(cuenta["Este"])     # 0, sin KeyError

### Un string que sabe convertirse

En los archivos del SMN un campo numérico puede venir como `"No se calcula"`.

### Un número con unidades

Heredando de `float` sirve para cuentas y además se muestra con su unidad.

In [ ]:
class Celsius(float):
    def __str__(self):
        return f"{float(self):.1f} °C"

    def __repr__(self):
        return f"Celsius({float(self)})"

    def a_fahrenheit(self):
        return self * 9 / 5 + 32


t = Celsius(8.5)
print(t)
print(t + 1, type(t + 1).__name__)   # la suma devuelve float común
print(t.a_fahrenheit())
print(max([Celsius(3), Celsius(8.5), Celsius(-1)]))